# 테일러 급수 — 곡선을 다항식으로 번역하기

> 미적분 11강 · 테일러 급수

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [테일러 급수 — 곡선을 다항식으로 번역하기](https://mioon1402.github.io/timeseriesdata/calc/C11-taylor.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 왜 다항식인가

## 1. 단계별로 닮아가기 — 값·기울기·곡률

## 2. 차수를 올리면 어디까지 따라오나

## 3. 팩토리얼은 왜 나오나

## 4. 수렴 반경 — 만능은 아니다

## 5. 공학에서의 쓰임 — 선형화

## 6. 파이썬으로 확인하기

**11-1. 차수를 올리며 cos 을 따라잡기**

In [ ]:
import numpy as np
from math import factorial

def 테일러_cos(x, n):
    """n차까지의 부분합"""
    return sum((-1)**(k//2) * x**k / factorial(k) for k in range(0, n+1, 2))

print(f"{'x':>5} {'참값 cos x':>14}", "".join(f"{f'{n}차':>13}" for n in [2, 4, 6, 8, 10]))
for x in [0.5, 1.0, 2.0, 3.0]:
    print(f"{x:>5} {np.cos(x):>14.8f}",
          "".join(f"{테일러_cos(x, n):>13.6f}" for n in [2, 4, 6, 8, 10]))

print("\n→ x 가 클수록 더 높은 차수가 필요하다.")

**11-2. 계수가 나오는 원리 — n번 미분**

In [ ]:
from math import factorial
import numpy as np

def n계도함수(f, x, n, h=1e-3):
    """중심차분을 n번 반복 (교육용 — 정확도는 낮다)"""
    if n == 0: return f(x)
    return (n계도함수(f, x+h, n-1, h) - n계도함수(f, x-h, n-1, h)) / (2*h)

print(f"{'n':>3} {'cos⁽ⁿ⁾(0) (수치)':>18} {'/ n!':>14} {'실제 계수':>14}")
실제 = [1, 0, -1/2, 0, 1/24, 0, -1/720]
for n in range(7):
    d = n계도함수(np.cos, 0.0, n)
    print(f"{n:>3} {d:>18.6f} {d/factorial(n):>14.8f} {실제[n]:>14.8f}")

print("\n→ 계수 = (n번 미분한 값) ÷ n!.  n! 은 xⁿ 을 n번 미분할 때 튀어나오는 값을 상쇄한다.")

**11-3. 유효 범위가 얼마나 넓어지나**

In [ ]:
import numpy as np
from math import factorial

def 유효범위(n, 허용=1e-3, 상한=12.0):
    """|테일러 - cos| < 허용 인 가장 큰 x"""
    x = np.linspace(0, 상한, 200001)
    오차 = np.abs(sum((-1)**(k//2) * x**k / factorial(k) for k in range(0, n+1, 2)) - np.cos(x))
    나쁜 = np.where(오차 >= 허용)[0]
    return 상한 if len(나쁜) == 0 else x[나쁜[0]]

print(f"{'차수':>6} {'오차 < 0.001 인 범위':>22}")
for n in [2, 4, 6, 8, 10, 14, 20]:
    print(f"{n:>6} {'±' + f'{유효범위(n):.3f}':>22}")

print("\n→ 차수를 올릴수록 '유효 사거리' 가 넓어진다. cos 은 수렴 반경이 무한이라 끝없이 넓힐 수 있다.")

**11-4. 수렴 반경 — 1/(1−x)**

In [ ]:
import numpy as np

def 등비부분합(x, n):
    return sum(x**k for k in range(n+1))

print(f"{'x':>6} {'참값 1/(1-x)':>16}", "".join(f"{f'{n}항':>14}" for n in [5, 10, 20, 40]))
for x in [0.5, 0.9, 0.99, 1.2]:
    참 = 1/(1-x) if x != 1 else float('inf')
    print(f"{x:>6} {참:>16.6f}", "".join(f"{등비부분합(x, n):>14.4f}" for n in [5, 10, 20, 40]))

print("\n→ |x| < 1 이면 수렴, |x| > 1 이면 항을 늘릴수록 더 크게 발산한다.")
print("  x=0.99 는 수렴하지만 아주 느리다 — 반경 경계에 가까울수록 느려진다.")

**11-5. 실수에서 멀쩡한데 반경이 1인 함수**

In [ ]:
import numpy as np

# 1/(1+x²) 의 급수: 1 - x² + x⁴ - x⁶ + ...
def 급수(x, n):
    return sum((-1)**k * x**(2*k) for k in range(n+1))

print("1/(1+x²) — 실수 어디서도 무한대가 되지 않는다. 그런데…\n")
print(f"{'x':>6} {'참값':>14} {'10항':>16} {'30항':>16}")
for x in [0.5, 0.9, 1.1, 1.5]:
    print(f"{x:>6} {1/(1+x**2):>14.8f} {급수(x, 10):>16.4f} {급수(x, 30):>16.4f}")

print("\n→ |x| > 1 에서 발산한다. 실수 그래프만 봐서는 이유를 알 수 없다.")
print("  복소수 x = ±i 에서 1+x² = 0 이 되어 특이점이 있기 때문이다.")
print("  중심 0 에서 ±i 까지의 거리가 1 — 그것이 수렴 반경이다.")

**11-6. sin θ ≈ θ 는 언제까지 통하나**

In [ ]:
import numpy as np

print(f"{'θ (rad)':>10} {'θ (도)':>10} {'sin θ':>14} {'상대 오차':>12}")
for θ in [0.01, 0.1, 0.3, 0.5, 1.0, 1.5]:
    오차 = abs(θ - np.sin(θ)) / np.sin(θ)
    print(f"{θ:>10} {np.degrees(θ):>10.1f} {np.sin(θ):>14.8f} {100*오차:>11.3f}%")

print("\n→ 5.7°(0.1rad) 에서 0.17%, 57°(1rad) 에서 18%.")
print("  '작은 진동' 가정이 깨지면 진자의 주기 공식이 통째로 틀린다.")

**11-7. sympy 로 급수 뽑기**

In [ ]:
import sympy as sp

x = sp.Symbol('x')
for 식 in [sp.cos(x), sp.sin(x), sp.exp(x), sp.log(1+x), 1/(1-x), sp.tan(x)]:
    print(f"{str(식):>12} → {sp.series(식, x, 0, 8)}")

**11-8. 연습문제**

In [ ]:
# 문제 1. eˣ 의 5차 근사로 e = e¹ 을 계산하고 참값과 비교하세요.

# 문제 2. tan x 의 3차 근사는 x + x³/3 입니다.
#         8강 연습문제의 lim (tan x - x)/x³ = 1/3 이 왜 그런지 설명해보세요.

# 문제 3. √(1+x) ≈ 1 + x/2 - x²/8 입니다. √1.1 을 이 식으로 계산하고
#         참값과 비교하세요.

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
import numpy as np
from math import factorial

# 문제 1
근사 = sum(1**k / factorial(k) for k in range(6))
print(f"문제 1: 5차 근사 {근사:.10f}   참값 e = {np.e:.10f}   오차 {abs(근사-np.e):.2e}")
print(f"        10차까지 가면 {sum(1/factorial(k) for k in range(11)):.10f}\n")

# 문제 2
print("문제 2: tan x ≈ x + x³/3 이므로 tan x - x ≈ x³/3")
print("        따라서 (tan x - x)/x³ ≈ 1/3.  로피탈 세 번보다 훨씬 빠르다")
for x in [0.1, 0.01]:
    print(f"        x={x}: {(np.tan(x)-x)/x**3:.8f}")
print()

# 문제 3
x = 0.1
근사3 = 1 + x/2 - x**2/8
print(f"문제 3: 근사 {근사3:.10f}   참값 √1.1 = {np.sqrt(1.1):.10f}")
print(f"        오차 {abs(근사3-np.sqrt(1.1)):.2e}  ← 소수 넷째 자리까지 맞는다")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)